# 03 — Pipeline Real-time & Dự báo Xu hướng — TechJobAI

**Notebook 3/3.** Trình bày cách hệ thống chạy **real-time**: lấy dữ liệu việc làm trực tiếp
từ các API công khai, phân tích tức thời, so sánh với baseline lịch sử (Kaggle),
và **dự báo xu hướng tuyển dụng** bằng hồi quy đa thức.

## Kiến trúc

```
                     PROVIDER CHAIN (backend/services/trend_service.py)
   ┌──────────────┐   fail   ┌──────────┐   fail   ┌────────────┐   fail   ┌──────────┐
   │ freehire.dev │ ───────► │ RemoteOK │ ───────► │ cache <1h  │ ───────► │ snapshot │
   └──────────────┘          └──────────┘          └────────────┘          └──────────┘
          │ live                   │ live                │ cache                │ demo
          └────────────┬───────────┴─────────────────────┴──────────────────────┘
                       ▼
        Analytics: top skills 24h/7d • top locations • top companies
        hiring velocity • Historical-vs-Realtime comparison + auto conclusion
                       ▼
        API: /api/realtime/report, /api/realtime/skills, /api/trending, ...
        Forecast: /api/realtime-trends  (Kaggle baseline + live count → Polynomial Regression)
```

**Vì sao cần provider chain?** API miễn phí bị rate-limit thường xuyên. Chuỗi fallback
đảm bảo dashboard **luôn** có dữ liệu: gọi live được thì cache lại; không được thì dùng
cache (<1 giờ); hết cách mới rơi về snapshot demo. Nguồn dữ liệu thật sự đang dùng được
gắn badge `Live / Cached / Snapshot` trên dashboard.

**MLOps:** `auto_worker.py` dùng thư viện `schedule` để retrain toàn bộ model
mỗi Chủ nhật 02:00 (chạy `retrain_all.py`).

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')

# Cho phép import package backend từ notebook
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from backend.services import trend_service

---
## 1. Gọi pipeline real-time

`build_realtime_report()` chạy toàn bộ provider chain và trả về bundle dữ liệu
mà tab **Real-time** trên dashboard sử dụng.

In [ ]:
report = trend_service.build_realtime_report()

print(f"Source        : {report.get('source')}   (live = gọi API thành công ngay lúc này)")
print(f"Total jobs    : {report.get('total_jobs')}")
print(f"Note          : {report.get('meta', {}).get('note', '')}")
print()
print('Top skills (7 ngày):')
for s in (report.get('top_skills_7d') or report.get('top_skills') or [])[:8]:
    print(f"  {s['skill']:<20} {s['count']:>3} jobs  ({s['share']}%)")

---
## 2. So sánh Historical (Kaggle) vs Real-time

Baseline lịch sử = tần suất kỹ năng tính từ `data/it_jobs_processed.csv` (Notebook 1).
So với tần suất kỹ năng real-time → hệ thống tự sinh **kết luận** về kỹ năng đang tăng nhiệt.

In [ ]:
comp = report.get('comparison', {})
rows = pd.DataFrame(comp.get('rows', []))
print('Kết luận tự động:', comp.get('conclusion', '—'))
rows

In [ ]:
if len(rows):
    x = np.arange(len(rows))
    plt.figure(figsize=(11, 5))
    plt.bar(x - 0.2, rows['historical_share'], width=0.4, label='Historical (Kaggle)', color='#9966ff')
    plt.bar(x + 0.2, rows['realtime_share'], width=0.4, label='Real-time', color='#4bc0c0')
    plt.xticks(x, rows['skill'].str.replace('_', ' ').str.title(), rotation=30, ha='right')
    plt.ylabel('Share (%)'); plt.title('Historical vs Real-time skill share')
    plt.legend(); plt.tight_layout(); plt.show()

---
## 3. Hiring velocity — tốc độ đăng tin 7 ngày gần nhất

In [ ]:
vel = pd.DataFrame(report.get('hiring_velocity', []))
if len(vel):
    plt.figure(figsize=(10, 4))
    plt.plot(vel['date'].str[5:], vel['count'], 'o-', color='#ff9f40')
    plt.fill_between(range(len(vel)), vel['count'], alpha=0.2, color='#ff9f40')
    plt.title('Việc làm mới / ngày (7 ngày)'); plt.ylabel('Jobs')
    plt.tight_layout(); plt.show()
vel

---
## 4. Dự báo xu hướng — logic của `/api/realtime-trends`

Bốn bước (code dưới tái hiện đúng logic trong `backend/server.py`):

1. **Baseline**: `data/backup_trends.csv` — 30 tháng, là **CHIẾU tăng trưởng** từ snapshot
   Kaggle 01/2024 + ~3.5%/năm (US BLS). *Không phải* số đo lịch sử thật (dataset Kaggle
   chỉ crawl trong 1 tuần) — hệ thống ghi rõ điều này trong response (`baseline_note`).
2. **Điểm real-time**: tổng số job từ provider chain, scale về độ lớn baseline, gắn vào tháng hiện tại
3. **Backtest chọn model**: giữ lại 6 điểm cuối làm holdout → so MAPE của
   Polynomial(bậc 2) vs **Holt-Winters** (statsmodels) → model thắng refit trên toàn chuỗi
4. **Dự báo 2 tháng** + khoảng tin cậy ±1.96σ (σ = độ lệch chuẩn residual backtest)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# (1) Baseline + (2) điểm realtime
df_trend = pd.read_csv(os.path.join(BASE_DIR, 'data', 'backup_trends.csv'))
data_points = df_trend.to_dict('records')
job_count = report.get('total_jobs', 0)
current_month = pd.Timestamp.now().strftime('%Y-%m')
if job_count > 0:
    avg_hist = sum(d['job_count'] for d in data_points) / len(data_points)
    scaled = int(job_count * max(avg_hist / max(job_count, 1), 0.1))
    if data_points[-1]['month'] == current_month:
        data_points[-1]['job_count'] = scaled
    else:
        data_points.append({'month': current_month, 'job_count': scaled})

y = np.array([d['job_count'] for d in data_points], dtype=float)

def poly_forecast(train, steps):
    X = np.arange(len(train)).reshape(-1, 1)
    poly = PolynomialFeatures(degree=2)
    m = LinearRegression().fit(poly.fit_transform(X), train)
    nxt = np.arange(len(train), len(train) + steps).reshape(-1, 1)
    return m.predict(poly.transform(nxt))

def holt_forecast(train, steps):
    m = ExponentialSmoothing(train, trend='add', seasonal=None,
                             initialization_method='estimated').fit()
    return np.asarray(m.forecast(steps))

# (3) Rolling-origin backtest: giữ 6 điểm cuối làm holdout
holdout = 6
train, test = y[:-holdout], y[-holdout:]
results = {}
for name, fn in [('polynomial_deg2', poly_forecast), ('holt_winters', holt_forecast)]:
    pred = fn(train, holdout)
    mape = float(np.mean(np.abs((test - pred) / np.maximum(test, 1))) * 100)
    results[name] = {'pred': pred, 'mape': round(mape, 2)}
    print(f'{name:<18} MAPE trên 6 điểm holdout = {mape:.2f}%')

model_used = min(results, key=lambda k: results[k]['mape'])
print(f'\n>>> Model thắng backtest: {model_used}')

# (4) Refit trên toàn chuỗi + dự báo 2 tháng + CI
fn = poly_forecast if model_used == 'polynomial_deg2' else holt_forecast
preds = fn(y, 2)
resid = test - results[model_used]['pred']
ci = float(1.96 * np.std(resid))

last_ts = pd.to_datetime(data_points[-1]['month'])
forecast = []
for p in preds:
    last_ts = last_ts + pd.DateOffset(months=1)
    forecast.append({'month': last_ts.strftime('%Y-%m'), 'job_count': int(max(0, p)),
                     'ci_low': int(max(0, p - ci)), 'ci_high': int(max(0, p + ci))})
print('Dự báo:', forecast)

In [ ]:
# Vẽ chart historical + forecast (giống chart "Dự Báo Xu Hướng" trên dashboard)
labels = [d['month'] for d in data_points] + [f['month'] for f in forecast]
actual = [d['job_count'] for d in data_points] + [None] * len(forecast)
fc = [None] * (len(data_points) - 1) + [data_points[-1]['job_count']] + [f['job_count'] for f in forecast]

plt.figure(figsize=(13, 5))
plt.plot(labels, actual, 'o-', color='#36a2eb', label='Số lượng jobs (thực tế)')
plt.plot(labels, fc, 'o--', color='#ffce56', label='Dự báo (Polynomial deg=2)')
plt.xticks(rotation=45, fontsize=8)
plt.ylabel('Job count'); plt.title('IT Hiring Trend — Historical + Real-time + Forecast')
plt.legend(); plt.tight_layout(); plt.show()

---
## 4c. Chuỗi dữ liệu TỰ TÍCH LŨY — `data/realtime_history.csv`

Vì Kaggle chỉ là snapshot 1 tuần, xu hướng thời gian **thật** phải do hệ thống tự xây:
- Mỗi lần pipeline realtime fetch thành công (live/cache), `history_service.record_snapshot()`
  ghi 1 dòng/giờ: timestamp, nguồn, tổng job, kỹ năng nóng nhất, phân bố domain.
- `auto_worker.py` chụp thêm 1 snapshot/ngày (08:00).
- Chạy hệ thống càng lâu → chuỗi càng dày → có thể fit mô hình chuỗi thời gian
  trên dữ liệu đo thật thay vì baseline chiếu.

In [ ]:
from backend.services import history_service

hist = history_service.load_history()
print(f'Số snapshot đã tích lũy: {len(hist)}')
if hist:
    dfh = pd.DataFrame(hist)
    display(dfh.tail(10))
    if len(dfh) >= 3:
        dfh['total_jobs'] = dfh['total_jobs'].astype(int)
        plt.figure(figsize=(11, 4))
        plt.plot(dfh['timestamp'], dfh['total_jobs'], 'o-', color='#4bc0c0')
        plt.xticks(rotation=45, fontsize=8)
        plt.title('Realtime jobs — chuỗi tự tích lũy'); plt.tight_layout(); plt.show()
else:
    print('Chưa có snapshot — chạy server/auto_worker để hệ thống bắt đầu tích lũy.')

---
## 5. Vận hành

| Thành phần | Cách chạy |
|------------|-----------|
| Server API + dashboard | `python -m backend.server` → http://localhost:5000 |
| Retrain thủ công | `python retrain_all.py` |
| MLOps tự động | `python auto_worker.py` (retrain Chủ nhật 02:00 hằng tuần) |
| Docker | `docker compose up --build` |

**Các endpoint real-time:**

| Endpoint | Trả về |
|----------|--------|
| `GET /api/realtime/report` | bundle đầy đủ cho tab Real-time |
| `GET /api/realtime/skills` | top skills 24h/7d + so sánh historical |
| `GET /api/realtime/trends` | hiring velocity + emerging roles |
| `GET /api/realtime-trends` | chuỗi lịch sử + dự báo 2 tháng (chart Dashboard) |
| `GET /api/trending` | view gọn {top_skills, top_jobs, top_locations} |
| `POST /api/historical_trends` | xu hướng lương/nhu cầu theo domain+bang (tab Xu Hướng Lịch Sử) |